# Notebook 7: Training Pipeline - JAX

In [ ]:
import jax
import jax.numpy as jnp
from flax import linen as nn
from flax.training import train_state
from jax import random
import optax

class SimpleModel(nn.Module):
    """Simple model for training demo"""
    vocab_size: int
    hidden_size: int
    
    def setup(self):
        self.embedding = nn.Embed(self.vocab_size, self.hidden_size)
        self.dense = nn.Dense(self.hidden_size)
        self.lm_head = nn.Dense(self.vocab_size)
    
    def __call__(self, input_ids, training=False):
        x = self.embedding(input_ids)
        x = self.dense(x)
        return self.lm_head(x)

def compute_loss(params, model, input_ids, labels):
    """Compute cross-entropy loss"""
    logits = model.apply(params, input_ids)
    
    # Shift for next token prediction
    logits = logits[:, :-1, :].reshape(-1, logits.shape[-1])
    labels = labels[:, 1:].reshape(-1)
    
    # Cross entropy
    loss = optax.softmax_cross_entropy_with_integer_labels(logits, labels)
    return jnp.mean(loss)

# Training step
def train_step(state, batch):
    input_ids, labels = batch
    
    def loss_fn(params):
        return compute_loss(state.params, model, input_ids, labels)
    
    loss, grads = jax.value_and_grad(loss_fn)(state.params)
    state = state.apply_gradients(grads=grads)
    
    return state, loss

# Setup
model = SimpleModel(vocab_size=50000, hidden_size=256)
rng = random.PRNGKey(42)
input_ids = random.randint(rng, (4, 32), 0, 50000)
labels = random.randint(rng, (4, 32), 0, 50000)

params = model.init(rng, input_ids)

tx = optax.adamw(learning_rate=1e-4)
state = train_state.TrainState.create(
    apply_fn=model.apply,
    params=params,
    tx=tx
)

# Training loop
for epoch in range(3):
    state, loss = train_step(state, (input_ids, labels))
    print(f"Epoch {epoch}: Loss = {loss:.4f}")

## Verification
1. ✅ Can implement training loop in JAX
2. ✅ Can use optax optimizer
3. ✅ Understand JAX functional training